# PrecisionSkin - Lesion Detector (boxes)

Trains a **separate** YOLOv8 object detector that draws boxes on lesions. It does **not** touch or retrain the
`facial_only` classifier - the classifier keeps giving the class scores, this model only adds *where*.

**What you need first:** Roboflow datasets exported as **YOLOv8** (folder with `data.yaml`, `train/`, `valid/`, `test/`),
and/or a COCO `_annotations.coco.json` + its images (your hyperpigmentation export works).

**Run order:** 1 install -> 2 config (edit paths) -> 3 build dataset -> 4 stats/preview -> 5 train -> 6 evaluate -> 7 export ONNX.

Notes for the paper: keep each source's licence/citation (Roboflow shows it on the project page); report test-set
precision / recall / mAP, not just pictures; this is a research prototype with limited data.
Ultralytics YOLO is AGPL-3.0 licensed - fine for an academic project, but cite it.

In [ ]:
# 1. install (run once). On Kaggle/Colab keep the preinstalled torch; on a local venv install torch with CUDA first.
%pip install -q ultralytics onnx onnxruntime pyyaml opencv-python-headless matplotlib pandas

In [ ]:
# 2. CONFIG - edit this cell only.
import os
from pathlib import Path

# Paths below are for running in WSL (/mnt/c/...). Windows: r"C:/Users/..."   Kaggle: "/kaggle/input/..."
WORK = Path("detector_work")            # everything generated goes here

CLASSES = ["acne", "hyperpigmentation"]   # final detector classes (add "eczema" only when you have boxed eczema data)

SOURCES = [
    # --- Roboflow YOLOv8 exports (folder containing data.yaml) ---
    # "to"      : the ONE detector class every kept box of this source becomes
    # "exclude" : source class names containing any of these words are dropped (lower-case substrings)
    {"name": "beautyai", "format": "yolo",                     # 1 class ('0' = acne), 1,598 unique photos
     "path": "/mnt/c/Users/Bautista/Downloads/ACNE Detection.v2i.yolov8",
     "to": "acne", "exclude": []},
    # Add more acne projects the same way. If a project has several classes, list the ones to DROP in "exclude":
    # {"name": "eastwest", "format": "yolo", "path": "/mnt/c/Users/Bautista/Downloads/detector_dataset/EastWest",
    #  "to": "acne", "exclude": ["scar", "pih", "non acne", "non_acne", "normal", "mole", "nevus"]},

    # --- COCO export (your hyperpigmentation file; images are in the same folder) ---
    {"name": "hyperpig", "format": "coco",
     "path": "/mnt/c/Users/Bautista/Downloads/PrecisionSkin/Classes/Hyperpigmentation",
     "json": "_annotations.coco(1).json",
     "to": "hyperpigmentation", "exclude": []},
    # Second hyperpigmentation export (YOLOv8 format) - paths that do not exist are skipped automatically:
    {"name": "hyperpig_yolo", "format": "yolo",
     "path": "/mnt/c/Users/Bautista/Downloads/detector_dataset/hyperpig_yolo",
     "to": "hyperpigmentation", "exclude": []},
]

MODEL = "yolov8s.pt"      # yolov8n.pt = fastest, yolov8s.pt = better for small spots, yolov8m.pt = best but slower
IMGSZ = 640               # try 800 or 960 if acne spots are tiny and you have GPU memory
EPOCHS = 100
PATIENCE = 25             # stop early if validation does not improve
SPLIT = (0.8, 0.1, 0.1)   # train / val / test, split by SOURCE PHOTO so augmented copies never leak across splits
SEED = 42
CONF = 0.25               # default confidence threshold used for previews and the exported metadata

In [ ]:
# 3. BUILD ONE CLEAN DATASET  (merge sources -> map classes -> drop duplicates -> leak-free split)
import json, re, shutil, hashlib, random, collections, yaml
from pathlib import Path

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLS_ID = {c: i for i, c in enumerate(CLASSES)}


def group_key(stem):
    """Roboflow names augmented copies 'photo_jpg.rf.<hash>' - all copies of one photo share this key.
    The key is NOT prefixed by the source, so the same photo exported by two projects stays in ONE split."""
    s = stem.split(".rf.")[0].lower()
    return re.sub(r"(_(jpe?g|png|webp))+$", "", s)


def keep_name(name, src):
    n = str(name).lower()
    if any(x in n for x in src.get("exclude", [])):
        return False
    inc = src.get("include")
    return True if not inc else any(x in n for x in inc)


def to_yolo_box(x0, y0, x1, y1, w, h):
    x0, x1 = max(0, min(x0, w)), max(0, min(x1, w))
    y0, y1 = max(0, min(y0, h)), max(0, min(y1, h))
    if x1 - x0 < 2 or y1 - y0 < 2:
        return None
    return ((x0 + x1) / 2 / w, (y0 + y1) / 2 / h, (x1 - x0) / w, (y1 - y0) / h)


records, dropped = [], collections.Counter()


def read_yolo_source(src):
    root = Path(src["path"])
    cfg = yaml.safe_load(open(root / "data.yaml", encoding="utf-8"))
    names = cfg["names"]
    names = {i: n for i, n in enumerate(names)} if isinstance(names, list) else {int(k): v for k, v in names.items()}
    cid = CLS_ID[src["to"]]
    for split in ("train", "valid", "val", "test"):
        imgdir = root / split / "images"
        if not imgdir.exists():
            continue
        for ip in sorted(imgdir.iterdir()):
            if ip.suffix.lower() not in IMG_EXT:
                continue
            lp = root / split / "labels" / (ip.stem + ".txt")
            boxes = []
            if lp.exists():
                for line in lp.read_text().splitlines():
                    p = line.split()
                    if len(p) < 5:
                        continue
                    if not keep_name(names.get(int(float(p[0])), ""), src):
                        dropped[src["name"] + ": class excluded"] += 1
                        continue
                    v = [float(t) for t in p[1:]]
                    if len(v) == 4:                       # box: cx cy w h (normalised)
                        xc, yc, bw, bh = v
                    else:                                  # polygon export -> enclosing box
                        xs, ys = v[0::2], v[1::2]
                        xc, yc, bw, bh = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
                    boxes.append((cid, xc, yc, bw, bh))
            records.append({"img": ip, "boxes": boxes, "source": src["name"], "group": group_key(ip.stem)})


def read_coco_source(src):
    root = Path(src["path"])
    j = json.load(open(root / src["json"], encoding="utf-8"))
    cats = {c["id"]: c["name"] for c in j["categories"]}
    imgs = {i["id"]: i for i in j["images"]}
    per = collections.defaultdict(list)
    for a in j["annotations"]:
        if keep_name(cats.get(a["category_id"], ""), src):
            per[a["image_id"]].append(a["bbox"])
    cid = CLS_ID[src["to"]]
    for iid, info in imgs.items():
        ip = root / info["file_name"]
        if not ip.exists():
            dropped[src["name"] + ": image file missing"] += 1
            continue
        w, h = info["width"], info["height"]
        boxes = []
        for x, y, bw, bh in per.get(iid, []):
            b = to_yolo_box(x, y, x + bw, y + bh, w, h)
            if b:
                boxes.append((cid,) + b)
        records.append({"img": ip, "boxes": boxes, "source": src["name"], "group": group_key(ip.stem)})


for s in SOURCES:
    assert s["to"] in CLS_ID, "source class " + s["to"] + " not in CLASSES"
    if not Path(s["path"]).exists():
        print("!! SKIPPED (path not found):", s["path"])
        continue
    (read_yolo_source if s["format"] == "yolo" else read_coco_source)(s)
    print("read", s["name"], "->", sum(r["source"] == s["name"] for r in records), "images")

# exact-duplicate removal across sources (same bytes = same photo)
seen, unique = {}, []
for r in records:
    h = hashlib.md5(r["img"].read_bytes()).hexdigest()
    if h in seen:
        dropped["exact duplicate image"] += 1
        continue
    seen[h] = 1
    unique.append(r)
records = unique

# keep images that have boxes; allow a few empty ones as background examples
withbox = [r for r in records if r["boxes"]]
empty = [r for r in records if not r["boxes"]]
random.Random(SEED).shuffle(empty)
keep_empty = max(0, len(withbox) // 20)
records = withbox + empty[:keep_empty]
dropped["no-box images not kept"] += len(empty) - keep_empty

# leak-free split: whole groups (all copies of one photo) go to one split
groups = sorted({r["group"] for r in records})
random.Random(SEED).shuffle(groups)
n = len(groups)
a = int(n * SPLIT[0])
b = int(n * (SPLIT[0] + SPLIT[1]))
split_of = {g: ("train" if i < a else "val" if i < b else "test") for i, g in enumerate(groups)}

DS = WORK / "dataset"
if DS.exists():
    shutil.rmtree(DS)
for sp in ("train", "val", "test"):
    (DS / sp / "images").mkdir(parents=True)
    (DS / sp / "labels").mkdir(parents=True)
for i, r in enumerate(records):
    sp = split_of[r["group"]]
    name = "%s_%06d" % (r["source"], i)
    shutil.copy2(r["img"], DS / sp / "images" / (name + r["img"].suffix.lower()))
    (DS / sp / "labels" / (name + ".txt")).write_text("".join("%d %.6f %.6f %.6f %.6f\n" % bx for bx in r["boxes"]))
    r["split"] = sp

DATA_YAML = DS / "data.yaml"
yaml.safe_dump({"path": str(DS.resolve()), "train": "train/images", "val": "val/images", "test": "test/images",
                "names": {i: c for i, c in enumerate(CLASSES)}}, open(DATA_YAML, "w"))
print("\ndataset written to", DS.resolve())
print("dropped:", dict(dropped))

In [ ]:
# 4. STATS + PREVIEW  - check this before training (box counts per class, box sizes, do the boxes sit on lesions?)
import pandas as pd, numpy as np, cv2, matplotlib.pyplot as plt

rows = []
for r in records:
    for (c, xc, yc, w, h) in r["boxes"]:
        rows.append({"split": r["split"], "class": CLASSES[c], "source": r["source"], "area": w * h})
df = pd.DataFrame(rows)
imgs = pd.DataFrame([{"split": r["split"], "source": r["source"]} for r in records])
print("IMAGES per split/source")
print(imgs.groupby(["split", "source"]).size().unstack(fill_value=0))
print("\nBOXES per split/class")
print(df.groupby(["split", "class"]).size().unstack(fill_value=0))
print("\nmedian box area as % of image, per class:")
print((df.groupby("class")["area"].median() * 100).round(2))
print("\nRule of thumb: >=1000 images and >=3000 boxes per class for a reasonable model. Tiny boxes (<0.1% of image) need IMGSZ 800-960.")

rng = random.Random(1)
sample = rng.sample(records, min(8, len(records)))
colors = {0: (0, 0, 255), 1: (255, 0, 0), 2: (0, 200, 255)}
fig, axs = plt.subplots(2, 4, figsize=(18, 9))
for ax, r in zip(axs.ravel(), sample):
    im = cv2.imread(str(r["img"]))
    H, W = im.shape[:2]
    for (c, xc, yc, w, h) in r["boxes"]:
        p0 = (int((xc - w / 2) * W), int((yc - h / 2) * H))
        p1 = (int((xc + w / 2) * W), int((yc + h / 2) * H))
        cv2.rectangle(im, p0, p1, colors.get(c, (0, 255, 0)), max(2, W // 300))
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(r["source"] + " | " + str(len(r["boxes"])) + " boxes")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 5. TRAIN
import torch
from ultralytics import YOLO

GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none (CPU training will be very slow)")

model = YOLO(MODEL)                       # COCO-pretrained weights: this is why a few hundred images can work
model.train(
    data=str(DATA_YAML), imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE,
    batch=-1 if GPU else 4, device=0 if GPU else "cpu",
    workers=0 if os.name == "nt" else 4, seed=SEED, cos_lr=True, close_mosaic=10,
    project=str((WORK / "runs").resolve()), name="lesion_detector", exist_ok=True,
    fliplr=0.5, flipud=0.0, degrees=10, scale=0.5,           # left/right flips and small rotation are safe for faces
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.35,                       # lighting / camera variation
)
SAVE_DIR = Path(model.trainer.save_dir)
BEST = SAVE_DIR / "weights" / "best.pt"
print("best weights:", BEST)

In [ ]:
# 6. EVALUATE on the held-out TEST split (never used for training or model selection) - report THESE numbers
from ultralytics import YOLO
m = YOLO(str(BEST))
metrics = m.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, conf=0.001, plots=True)

print("\nTEST SET  precision %.3f | recall %.3f | mAP@0.5 %.3f | mAP@0.5:0.95 %.3f" %
      (metrics.box.mp, metrics.box.mr, metrics.box.map50, metrics.box.map))
per_class = {}
for k, ci in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(k)
    per_class[CLASSES[int(ci)]] = {"precision": float(p), "recall": float(r), "mAP50": float(ap50), "mAP50_95": float(ap)}
    print("  %-18s P %.3f  R %.3f  mAP50 %.3f  mAP50-95 %.3f" % (CLASSES[int(ci)], p, r, ap50, ap))

# visual check on test images
test_imgs = sorted((DS / "test" / "images").iterdir())[:8]
res = m.predict([str(p) for p in test_imgs], conf=CONF, imgsz=IMGSZ, verbose=False)
fig, axs = plt.subplots(2, 4, figsize=(18, 9))
for ax, r in zip(axs.ravel(), res):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 6b. OPTIONAL - try it on YOUR webcam captures (the real use case). Set the glob to your captures folder.
import glob
CAPTURE_GLOB = "/mnt/c/Users/Bautista/AppData/Local/LUMYVUE/Captures/**/original.png"
files = sorted(glob.glob(CAPTURE_GLOB, recursive=True))[-8:]
if not files:
    print("no captures found - set CAPTURE_GLOB")
else:
    res = YOLO(str(BEST)).predict(files, conf=CONF, imgsz=IMGSZ, verbose=False)
    fig, axs = plt.subplots(2, 4, figsize=(18, 9))
    for ax, r in zip(axs.ravel(), res):
        ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# 7. EXPORT TO ONNX (+ metadata) so the C# app can load it like the classifier
import datetime, hashlib, onnxruntime as ort

onnx_src = Path(YOLO(str(BEST)).export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True, dynamic=False))
OUT = WORK / "export"
OUT.mkdir(exist_ok=True)
onnx_path = OUT / "lesion_detector.onnx"
shutil.copy2(onnx_src, onnx_path)

meta = {
    "model": "YOLO lesion detector", "weights": MODEL, "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "classes": CLASSES, "imgsz": IMGSZ, "default_conf": CONF, "nms_iou": 0.45,
    "onnx_sha256": hashlib.sha256(onnx_path.read_bytes()).hexdigest(),
    "test_metrics": {"precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
                     "mAP50": float(metrics.box.map50), "mAP50_95": float(metrics.box.map), "per_class": per_class},
    "sources": [{"name": s["name"], "format": s["format"], "to": s["to"]} for s in SOURCES],
    "split_images": {"train": int((imgs.split == "train").sum()), "val": int((imgs.split == "val").sum()), "test": int((imgs.split == "test").sum())},
    "preprocess": "letterbox to IMGSZxIMGSZ (pad 114), RGB, float32 / 255, NCHW",
    "output": "[1, 4+num_classes, N]: rows cx,cy,w,h (pixels of the letterboxed input) then class scores; no objectness; apply NMS",
    "note": "Research prototype. Boxes show candidate lesion locations; not a diagnosis.",
}
json.dump(meta, open(OUT / "lesion_detector.json", "w"), indent=2)

# sanity: run the ONNX file once
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
dummy = np.random.rand(1, 3, IMGSZ, IMGSZ).astype("float32")
out = sess.run(None, {inp.name: dummy})[0]
print("ONNX ok | input", inp.name, inp.shape, "| output", out.shape)
print("saved:", onnx_path.resolve())
print("saved:", (OUT / "lesion_detector.json").resolve())